## Código inicial

In [ ]:
import os

from_drive = True  # same flag you use everywhere

if os.environ.get("ATLAS_BOOTSTRAPPED") != "1":
    # ---------- GIT ON COLAB ONLY ----------
    try:
        from google.colab import userdata

        git_token = userdata.get('gitToken')
        git_user = userdata.get('gitUser')
        git_url = f'https://{git_token}@github.com/rene-aum/Atlas.git'
        branch_to_pull = 'dev'

        os.chdir('/content')

        if not os.path.isdir('Atlas'):
            !git clone {git_url}

        %cd Atlas
        !git fetch origin {branch_to_pull}
        !git checkout {branch_to_pull}
        !git pull origin {branch_to_pull}

        !pip install -r PipelinesConsumo/src/requirements.txt
        %cd PipelinesConsumo

    except Exception as e:
        print(e)
        print('Running in other environment not colab probably!')

    # ---------- DRIVE + SHEETS ----------
    if from_drive:
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        import gspread
        from google.auth import default
        from gspread_dataframe import set_with_dataframe
        import gdown

        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)

        creds, _ = default()
        gc = gspread.authorize(creds)

    os.environ["ATLAS_BOOTSTRAPPED"] = "1"
else:
    print("Bootstrap already done, assuming orchestrator ran it.")

In [ ]:
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
import sys
sys.path.append('..')
sys.path.append('../..')
from utils.utils import (get_dates_dataframe,
                       add_year_week,
                       custom_read,
                       process_columns)
from PipelinesConsumo.src.rawAtlas import RawAtlas
from PipelinesConsumo.src.processedAtlas import ProcessedAtlas
from src.transformed import Transformed
from utils.drive_toolbox import(from_drive_to_local,
                             get_last_modification_date_drive,
                             create_sheets_in_drive_folder,
                             update_sheets_in_drive_folder,
                             read_from_google_sheets,
                             list_file_ids_for_drive_folder,
                             create_csv_file_in_drive_folder,
                             write_csv_to_drive,
                             read_csv_from_drive,
                             send_google_chat_notification,
                             update_sheets_in_drive_folder_chunked)
from src.constants import (atlas_raw_output_folder_id,
                           atlas_consumo_output_folder_id,
                           consumo_sheets_ids_dict,
                           data_source_folder_id,
                           raw_output_ids,
                           folder_id_bauto_gabo,
                           id_reporte_ventas,
                           id_edas_referenciados
                           )

import time
warnings.filterwarnings('ignore')

# **Carga de df**

In [ ]:
import pandas as pd
import numpy as np

# --- SELECTOR DE PROYECTO ---
opcion = 8

# --- DICCIONARIO DE CONFIGURACIONES ---
proyectos = {
    1: {
        'nombre': 'AcClientes',
        'id_prod': r'1Re7omesAjvf8hoQ9n_BTCMBUnM37YOj6VOJTZHs3I6I',
        'id_dev':  r'1ppBSiQ52nSMiHNBMRwk70BFONHp_I6yZPHrFxVyEo_Q',
        'hoja': 'Hoja 1',
        'col_llave' : 'id_am',
        'hoja_escritura': 'AcClientes-Data'
    },
    2: {
        'nombre': 'AcPedidos',
        'id_prod': r'1aBfKoX6d9BEPhhYg-DDuvC5QFcd3EQtXb0YrU7n4nYM',
        'id_dev':  r'1yF9XA0PSdZLExQStGcHFSpUI56dJYE50aaTlFEV8uCk',
        'hoja': 'Hoja 1',
        'col_llave' : 'sf_order_id',
        'hoja_escritura': 'AcPedidos-Data'
    },
    3: {
        'nombre': 'AcVisitasUnicas',
        'id_prod': r'1jmbOj9a7-dnXUgH276lJ-blP-auDdWGemzia1RTsNEo',
        'id_dev':  r'1ELHlm0yWNsRn7kmKXpTQBhvSlvQeBYdTglPjG5DxKTQ',
        'hoja': 'Hoja 1',
        'col_llave' : 'date',
        'hoja_escritura': 'AcVisitasUnicas-Data'
    },
    4: {
        'nombre': 'AcPublicacionesCanceladas',
        'id_prod': r'1AZdaqfSw6QV9eXNRYgnuizvSsWRcyBUJ4IgsZkcw9gk',
        'id_dev':  r'1GUgdzRhmmec2L_HqJBVhheK2YUMreTQ1E5npTaE2fuY',
        'hoja': 'Hoja 1',
        'col_llave' : 'sku',
        'hoja_escritura': 'AcPublicacionesCanceladas-Data'
    },

    5 : {
        'nombre': 'AcPublicaciones',
        'id_prod': r'1NIvj4zjUO9N4fsiW1I85VI2RfTCT54nvbBON_l0xeYI',
        'id_dev':  r'18HOA3JS33OpICwY_7XgJpXh49bcn3qFLEPx723c1bd4',
        'hoja': 'Hoja 1',
        'col_llave' : 'sku',
        'hoja_escritura': 'AcPublicaciones-Data'
    },

### Batch 2

    6 : {
        'nombre': 'AcAdobeFunnelCompradorTotal',
        'id_prod': r'1XEThwBlrkC01n3ocAtLXuHdattooWKmK-yZEiYzkr-8',
        'id_dev':  r'1pLJVFQoHO5uUVXlPwkPiIKP1ugJjva-kvypMV_U4HBI',
        'hoja': 'Hoja 1',
        'col_llave' : 'date',
        'hoja_escritura': 'AcAdobeFunnelCompradorTotal-Data'
    },

    7 : {
        'nombre': 'AcAdobeFunnelCompradorUsuario',
        'id_prod': r'1iE0CLKpfje1BV42EH6vnj58nUes3T1RpVvB_4kXCawM',
        'id_dev':  r'1kwtOw5EhTnsjODtS7xnVqmp5R7O2VHYzV6dJu8hv59M',
        'hoja': 'Hoja 1',
        'col_llave' : 'date',
        'hoja_escritura': 'AcAdobeFunnelCompradorUsuario-Data'
    },

    8 : {
        'nombre': 'AcAdobeFunnelVendedorTotal',
        'id_prod': r'1prXQStwzaAMiPwSz24Fi2sfetuO24l6YyYA1Z_7xVpE',
        'id_dev':  r'1lfQdlkl_pYLrD6GFJrOWFJaFaLEZvQygAhAnkljxspg',
        'hoja': 'Hoja 1',
        'col_llave' : 'date',
        'hoja_escritura': 'AcAdobeFunnelVendedorTotal-Data'
    },

    9 : {
        'nombre': 'AcAdobeFunnelVendedorUsuario',
        'id_prod': r'15S67jk2ZwRjhpaicksWPMwk6HPV2sk4rtSEkLHjsTr8',
        'id_dev':  r'18NezvGLza7Xs1x0mLTzDf__qW3w0BS3u_IrFqtDeu0Q',
        'hoja': 'Hoja 1',
        'col_llave' : 'date',
        'hoja_escritura': 'AcAdobeFunnelVendedorUsuario-Data'
    }
}

# --- CARGA AUTOMÁTICA ---
if opcion in proyectos:
    config = proyectos[opcion]
    print(f"Cargando datos del proyecto: {config['nombre']}...")

    # Carga de datos
    prod = read_from_google_sheets(gc, config['id_prod'], config['hoja'])
    dev = read_from_google_sheets(gc, config['id_dev'], config['hoja'])

    # Forzar nombres de columnas a string para evitar errores en comparaciones posteriores
    prod.columns = [str(c) for c in prod.columns]
    dev.columns = [str(c) for c in dev.columns]

    col_llave = config['col_llave']
    hoja_escritura = config['hoja_escritura']
else:
    print(f"Error: La opción {opcion} no es válida.")

print("Datos cargados. Nombres de columnas normalizados a string.")

In [ ]:
import re
import pandas as pd

# ============================================================
# PIPELINE UNIFICADO DE LIMPIEZA
# ============================================================

def limpiar_pipeline_completo(df):
    """
    Pipeline unificado en un solo paso por celda:
    1. Diccionario de traducciones (primero, antes de que re-encode lo dañe)
    2. Re-encode de bytes (MacRoman/Latin-1 → UTF-8) como fallback
    3. Conversión de columnas numéricas a enteros
    """

    traducciones = {
        # Raíz (√)
        '√°': 'á', '√©': 'é', '√≠': 'í', '√≥': 'ó', '√∫': 'ú',
        '√±': 'ñ', '√º': 'ü', '√ﾁ': 'Á', '√ﾉ': 'É', '√ﾍ': 'Í',
        '√ﾓ': 'Ó', '√ﾚ': 'Ú', '√ﾑ': 'Ñ', '√Ë': 'Ñ', '√Â': 'É',
        '√Ç': 'Í', '√ç': 'í', '√Ì': 'Ó', '√Å': 'Á', '√Ö': 'Ú',
        # Ã + combinaciones
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã±': 'ñ', 'Ã"': 'Ó', 'Ã‰': 'É', 'Ã ': 'À', 'Ã‘': 'Ñ',
        'Ã\x9a': 'Ú', 'Ã\x93': 'Ó', 'Ã\x81': 'Á', 'Ã\x89': 'É', 'Ã\x8d': 'Í',
        'Ã“': 'Ó', 'ÃŠ': 'Ú',
        # U+00C5 — el culpable de JESÅS → JESÚS (codepoint directo, no el glifo)
        '\u00c5': 'Ú',
        # Especiales
        'â€"': '—', 'â€¦': '...', '\xa0': ' ',
        # Residuo — siempre al final por ser substring de otros
        'Â': '',
    }

    patron = re.compile(
        '|'.join(re.escape(k) for k in sorted(traducciones, key=len, reverse=True))
    )

    # Caracteres que el diccionario ya cubre y que re-encode puede dañar
    PROTEGIDOS = set('√ÃÂ\u00c5')

    def fix_completo(val):
        if not isinstance(val, str):
            return val

        # Paso 1: diccionario PRIMERO — cubre patrones conocidos sin riesgo
        val = patron.sub(lambda m: traducciones[m.group(0)], val)

        # Paso 2: re-encode solo si quedan caracteres sin resolver
        # y ninguno de los protegidos está presente (para no deshacer el paso 1)
        if any(c in val for c in ('√', 'Ã', 'â€')) and not any(c in val for c in PROTEGIDOS):
            for encoding in ('macroman', 'latin-1'):
                try:
                    candidato = val.encode(encoding).decode('utf-8')
                    if not re.search(r'[√Ã]|â€', candidato):
                        val = candidato
                        break
                except (UnicodeEncodeError, UnicodeDecodeError):
                    continue

        return val

    for col in df.columns:
        if df[col].dtype == 'object':
            mask = df[col].notna()
            df.loc[mask, col] = df[col].loc[mask].apply(fix_completo).str.strip()

        try:
            temp_num = pd.to_numeric(df[col], errors='coerce')
            if not temp_num.isna().all():
                df[col] = temp_num.fillna(0).round(0).astype(int)
        except:
            continue

    return df


# ============================================================
# HELPERS DE NORMALIZACIÓN
# ============================================================

def limpiar_llave_int(serie):
    def procesar_valor(x):
        if pd.isna(x) or str(x).strip().lower() in ['nan', 'none', '']:
            return "0"
        val_str = str(x).strip()
        try:
            f = float(val_str)
            if f.is_integer():
                return str(int(f))
            return str(f)
        except ValueError:
            return val_str
    return serie.apply(procesar_valor)


def forzar_int_str(valor):
    """Normaliza valores para una comparación justa entre Prod y Dev."""
    if pd.isna(valor) or str(valor).lower() in ['nan', 'none', '']:
        return ""
    try:
        f_val = float(valor)
        if f_val == int(f_val):
            return str(int(f_val))
        return str(f_val)
    except:
        return str(valor).strip()


# ============================================================
# EJECUCIÓN
# ============================================================

print("Iniciando limpieza de caracteres y normalización de tipos...")

prod = limpiar_pipeline_completo(prod)
dev  = limpiar_pipeline_completo(dev)

print("✅ Limpieza completada.")
print(f"Muestra PROD (billing_firstname): {prod['billing_firstname'].head(3).tolist() if 'billing_firstname' in prod.columns else 'Columna no encontrada'}")

In [ ]:
# --- NORMALIZACIÓN DE PAÍS (MÉXICO -> MX) CON PRESERVACIÓN DE NULOS ---

def normalizar_pais(df):
    """
    Estandariza 'MEXICO' a 'MX' en la columna country, manteniendo nulos como vacíos.
    """
    # Buscamos la columna (case-insensitive)
    target_col = next((c for c in df.columns if c.lower() == 'country'), None)

    if target_col:
        # 1. Convertimos a string pero mantenemos los NaN reales de Pandas
        # Usamos .str.upper() directamente; si es NaN, Pandas lo deja como NaN
        df[target_col] = df[target_col].str.upper().str.strip()

        # 2. Reemplazo específico
        # Agregamos 'NAN' a la lista por si acaso el df ya traía el string "nan"
        lista_mexico = ['MEXICO', 'MÉXICO', 'MÉXICO', 'Mexico']
        df[target_col] = df[target_col].replace(lista_mexico, 'MX')

        # 3. Limpieza de valores nulos/vacíos
        # Aseguramos que si quedó como 'NAN', None o np.nan, se mantenga como un string vacío o NaN real
        # En este caso, lo normalizamos a un string vacío '' para facilitar comparaciones
        df[target_col] = df[target_col].fillna('').replace(['NAN', 'NONE', 'N/A'], '')

        print(f"Columna '{target_col}' normalizada. Vacíos preservados.")
    else:
        print("La columna 'country' no existe. Saltando...")

    return df

# Aplicar a los dataframes actuales
print("Iniciando normalización de países...")

prod = normalizar_pais(prod)
dev = normalizar_pais(dev)

print("✅ Proceso de país completado.")

In [ ]:
# --- NORMALIZACIÓN DE TELÉFONO (ÚLTIMOS 10 DÍGITOS) ---

def normalizar_telefono(df):
    """
    Si existe la columna 'phone', extrae los 10 dígitos de la derecha
    eliminando prefijos y caracteres especiales.
    """
    # Buscamos la columna (case-insensitive)
    target_col = next((c for c in df.columns if c.lower() == 'phone'), None)

    if target_col:
        # 1. Convertir a string y limpiar nulos
        df[target_col] = df[target_col].fillna('').astype(str)

        # 2. Eliminar todo lo que no sea un número (guiones, +, espacios, etc.)
        # Esto evita que un '+52' cuente como caracteres si queremos solo números
        df[target_col] = df[target_col].str.replace(r'\D', '', regex=True)

        # 3. Tomar los 10 dígitos de la derecha
        # Si el número tiene menos de 10, se queda como está
        df[target_col] = df[target_col].str[-10:]

        # 4. Asegurar que los que quedaron vacíos no sean 'NAN' por el cast anterior
        df[target_col] = df[target_col].replace(['nan', 'None', 'NAN'], '')

        print(f"Columna '{target_col}' normalizada a 10 dígitos.")
    else:
        print("La columna 'phone' no existe. Saltando...")

    return df

# Aplicar a los dataframes actuales
print("Iniciando normalización de teléfonos...")

prod = normalizar_telefono(prod)
dev = normalizar_telefono(dev)

print("✅ Proceso de teléfono completado.")

In [ ]:
# --- NORMALIZACIÓN A MAYÚSCULAS ---

def normalizar_a_mayusculas(df):
    """
    Convierte todo el contenido de las columnas de texto a MAYÚSCULAS
    para asegurar comparaciones precisas (ej. 'mx' == 'MX').
    """
    # Seleccionamos solo las columnas que son de tipo objeto (strings)
    columnas_texto = df.select_dtypes(include=['object']).columns

    for col in columnas_texto:
        # Convertimos a string por seguridad, pasamos a mayúsculas y quitamos espacios
        df[col] = df[col].astype(str).str.upper().str.strip()

        # Opcional: Si el string es 'NAN' o 'NONE' debido al astype(str),
        # podrías querer devolverlo a un valor vacío real, pero para comparar
        # suele ser mejor dejarlo estandarizado.

    return df

# Aplicar a ambos DataFrames
print("Normalizando textos a MAYÚSCULAS...")

prod = normalizar_a_mayusculas(prod)
dev = normalizar_a_mayusculas(dev)

print("✅ Normalización terminada. Ahora 'mx' y 'MX' son idénticos.")

# *0. Validación directa*

In [ ]:
son_iguales = prod.equals(dev)

if son_iguales:
    print("Los DataFrames son idénticos.")
else:
    print("Hay diferencias entre los DataFrames.")

# *1. Validación de columnas*

*1.1 Nombre y cantidad de columnas*

1.2 Tipos de datos

In [ ]:
# 1. Creamos un DataFrame que une los tipos de datos de ambos
comparacion_tipos = pd.concat([prod.dtypes, dev.dtypes], axis=1, keys=['prod_type', 'dev_type'])

# 2. Creamos una columna booleana para ver si son iguales
comparacion_tipos['coinciden'] = comparacion_tipos['prod_type'] == comparacion_tipos['dev_type']

# 3. Separamos los resultados
diferencias = comparacion_tipos[comparacion_tipos['coinciden'] == False]

if diferencias.empty:
    print("Todos los tipos de dato coinciden.")
else:
    print("Se encontraron discrepancias en los tipos de datos:")
    print(diferencias)

# *2. Validación de filas*

2.1 Cantidad de filas

In [ ]:
# 1. Obtener el número de filas
filas_prod = len(prod)
filas_dev = len(dev)

# 2. Comparación de conteo
if filas_prod == filas_dev:
    print(f"Coincidencia exacta: Ambos tienen {filas_prod} filas.")
else:
    diferencia = abs(filas_prod - filas_dev)
    if filas_prod > filas_dev:
        print("Diferencia detectada: PROD tiene más registros.")
        print(f"Filas en PROD: {filas_prod}")
        print(f"Filas en DEV:  {filas_dev}")
        print(f"Diferencia: {diferencia} filas adicionales en PROD.")
    else:
        print("Diferencia detectada: DEV tiene más registros.")
        print(f"Filas en DEV:  {filas_dev}")
        print(f"Filas en PROD: {filas_prod}")
        print(f"Diferencia: {diferencia} filas adicionales en DEV.")

2.2 Existencia de duplicados

In [ ]:
from collections import Counter

# Función para limpiar la columna clave según su tipo
def limpiar_llave(df, columna):
    if pd.api.types.is_numeric_dtype(df[columna]):
        return df[columna].fillna(0).astype(int)
    return df[columna]

if opcion <= 5:
    ids_prod_clean = limpiar_llave(prod, col_llave)
    ids_dev_clean  = limpiar_llave(dev, col_llave)

    # 1. Validación de Duplicados
    dups_prod = ids_prod_clean.duplicated().sum()
    dups_dev  = ids_dev_clean.duplicated().sum()

    print(f"Duplicados en prod: {dups_prod}")
    print(f"Duplicados en dev:  {dups_dev}")

    if opcion == 4:
        ids_prod = Counter(ids_prod_clean)
        ids_dev  = Counter(ids_dev_clean)

        esta_contenido = all(ids_dev[k] >= v for k, v in ids_prod.items())

        solo_prod_counter = ids_prod - ids_dev
        solo_dev_counter  = ids_dev  - ids_prod

        solo_prod = list(solo_prod_counter.elements())
        solo_dev  = list(solo_dev_counter.elements())

        nombre_menor = "prod" if len(ids_prod_clean) <= len(ids_dev_clean) else "dev"
        nombre_mayor = "dev"  if nombre_menor == "prod" else "prod"

    else:
        ids_prod = set(ids_prod_clean)
        ids_dev  = set(ids_dev_clean)

        if len(ids_prod) <= len(ids_dev):
            nombre_menor, set_menor = "prod", ids_prod
            nombre_mayor, set_mayor = "dev",  ids_dev
        else:
            nombre_menor, set_menor = "dev",  ids_dev
            nombre_mayor, set_mayor = "prod", ids_prod

        esta_contenido = set_menor.issubset(set_mayor)

        solo_prod = list(ids_prod - ids_dev)
        solo_dev  = list(ids_dev  - ids_prod)

    print(f"\nResultados de pertenencia:")
    print(f"¿La totalidad de {nombre_menor} está contenida en {nombre_mayor}?: {esta_contenido}")
    print(f"\nRegistros en prod que no están en dev: {len(solo_prod)}")
    print(f"Registros en dev que no están en prod: {len(solo_dev)}")

    if solo_prod:
        print(f"Ejemplos de IDs solo en prod: {sorted(solo_prod)[:5]}")
    if solo_dev:
        print(f"Ejemplos de IDs solo en dev: {sorted(solo_dev)[:5]}")

else:
    # ============================================================
    # OPCIONES > 5: LLAVE COMPUESTA (date + id_am)
    # Duplicados conservados — lógica tolerante como opción 4
    # Vacíos en id_am: cualquier vacío del mismo día matchea con cualquier otro
    # ============================================================

    COL_DATE = 'date'
    COL_IDAM = 'id_am'

    def preparar_df(df):
        df = df.copy()
        df[COL_DATE] = df[COL_DATE].astype(str).str.strip()
        df[COL_IDAM] = df[COL_IDAM].apply(forzar_int_str)  # vacíos → ''
        return df

    prod = preparar_df(prod)
    dev  = preparar_df(dev)

    # ----------------------------------------------------------
    # 1. SEPARAR CON id_am PRESENTE VS VACÍO
    # ----------------------------------------------------------
    prod_con = prod[prod[COL_IDAM] != ''].copy()
    prod_vac = prod[prod[COL_IDAM] == ''].copy()
    dev_con  = dev[dev[COL_IDAM]  != ''].copy()
    dev_vac  = dev[dev[COL_IDAM]  == ''].copy()

    def llave_str(df):
        return df[COL_DATE].astype(str) + '||' + df[COL_IDAM].astype(str)

    # Counter para respetar duplicados (igual que opción 4)
    counter_prod_con = Counter(llave_str(prod_con))
    counter_dev_con  = Counter(llave_str(dev_con))
    counter_prod_vac = Counter(prod_vac[COL_DATE])
    counter_dev_vac  = Counter(dev_vac[COL_DATE])

    # Validación de duplicados (informativo)
    llaves_prod_todas = llave_str(prod_con).tolist() + [f"{d}||" for d in prod_vac[COL_DATE]]
    llaves_dev_todas  = llave_str(dev_con).tolist()  + [f"{d}||" for d in dev_vac[COL_DATE]]
    dups_prod = pd.Series(llaves_prod_todas).duplicated().sum()
    dups_dev  = pd.Series(llaves_dev_todas).duplicated().sum()
    print(f"Duplicados en prod: {dups_prod}")
    print(f"Duplicados en dev:  {dups_dev}")

    registros_errores = []

    # ----------------------------------------------------------
    # 2. IDENTIFICAR ÚNICOS (EXISTENCIA) — respetando duplicados
    # ----------------------------------------------------------

    # --- Con id_am presente ---
    solo_prod_con = list((counter_prod_con - counter_dev_con).elements())
    solo_dev_con  = list((counter_dev_con  - counter_prod_con).elements())

    for llave in solo_prod_con:
        registros_errores.append({'llave': llave, 'Error': 'SOLO_EN_PROD', 'Flag': 1})
    for llave in solo_dev_con:
        registros_errores.append({'llave': llave, 'Error': 'SOLO_EN_DEV', 'Flag': 1})

    # --- Vacíos: comparar conteo por día ---
    solo_prod_vac = list((counter_prod_vac - counter_dev_vac).elements())
    solo_dev_vac  = list((counter_dev_vac  - counter_prod_vac).elements())

    for dia in solo_prod_vac:
        registros_errores.append({'llave': f"{dia}||VACIO_EXTRA", 'Error': 'SOLO_EN_PROD', 'Flag': 1})
    for dia in solo_dev_vac:
        registros_errores.append({'llave': f"{dia}||VACIO_EXTRA", 'Error': 'SOLO_EN_DEV', 'Flag': 1})

    # Resumen de pertenencia
    total_solo_prod = len(solo_prod_con) + len(solo_prod_vac)
    total_solo_dev  = len(solo_dev_con)  + len(solo_dev_vac)
    esta_contenido  = total_solo_prod == 0

    nombre_menor = "prod" if len(llaves_prod_todas) <= len(llaves_dev_todas) else "dev"
    nombre_mayor = "dev"  if nombre_menor == "prod" else "prod"

    print(f"\nResultados de pertenencia:")
    print(f"¿La totalidad de {nombre_menor} está contenida en {nombre_mayor}?: {esta_contenido}")
    print(f"\nRegistros en prod que no están en dev: {total_solo_prod}")
    print(f"Registros en dev que no están en prod: {total_solo_dev}")

    if solo_prod_con:
        print(f"Ejemplos de llaves solo en prod: {sorted(solo_prod_con)[:5]}")
    if solo_dev_con:
        print(f"Ejemplos de llaves solo en dev:  {sorted(solo_dev_con)[:5]}")

2.3 Comparación de contenido

In [ ]:
# --- COMPARACIÓN DE DATOS Y DETECCIÓN DE DIFERENCIAS ---

if opcion <= 5:
  # 1. IDENTIFICACIÓN DE DIFERENCIAS DE EXISTENCIA (ÚNICOS)
  ids_prod_full = set(limpiar_llave_int(prod[col_llave]))
  ids_dev_full  = set(limpiar_llave_int(dev[col_llave]))

  registros_errores = []

  ids_solo_prod = ids_prod_full - ids_dev_full
  ids_solo_dev  = ids_dev_full  - ids_prod_full

  for id_val in ids_solo_prod:
      registros_errores.append({col_llave: id_val, 'Error': 'SOLO_EN_PROD', 'Flag': 1})
  for id_val in ids_solo_dev:
      registros_errores.append({col_llave: id_val, 'Error': 'SOLO_EN_DEV',  'Flag': 1})

  # 2. ALINEACIÓN PARA COMPARACIÓN DE CONTENIDO
  ids_comunes = ids_prod_full & ids_dev_full

  prod_comun = prod[limpiar_llave_int(prod[col_llave]).isin(ids_comunes)].copy()
  dev_comun  = dev[limpiar_llave_int(dev[col_llave]).isin(ids_comunes)].copy()

  prod_comun[col_llave] = limpiar_llave_int(prod_comun[col_llave])
  dev_comun[col_llave]  = limpiar_llave_int(dev_comun[col_llave])

  columnas_a_comparar = [c for c in prod.columns if c != col_llave and c in dev.columns]

  print(f"Analizando {len(ids_comunes)} registros comunes y {len(ids_solo_prod) + len(ids_solo_dev)} únicos.")
  print(f"{'COLUMNA':<30} | {'ESTADO':<20} | {'DETALLE'}")
  print("-" * 110)

  # 3. MERGE Y COMPARACIÓN DE COLUMNAS
  if opcion == 4:
      # Cross join por llave: todas las combinaciones posibles entre prod y dev
      comparativo = pd.merge(prod_comun, dev_comun, on=col_llave, suffixes=('_prod', '_dev'), how='inner')

      for col in columnas_a_comparar:
          cp, cd = f"{col}_prod", f"{col}_dev"

          v_prod_norm = comparativo[cp].apply(forzar_int_str)
          v_dev_norm  = comparativo[cd].apply(forzar_int_str)

          # Para duplicados: una llave está "OK" si existe AL MENOS UNA combinación que coincide
          coincide      = v_prod_norm == v_dev_norm
          ids_con_match = set(comparativo.loc[coincide, col_llave])

          # Solo_prod_tiene / solo_dev_tiene se evalúan por combinación, luego se filtra
          solo_prod_tiene = comparativo[cp].notna() & (comparativo[cp] != '') & \
                            (comparativo[cd].isna() | (comparativo[cd] == ''))
          solo_dev_tiene  = comparativo[cd].notna() & (comparativo[cd] != '') & \
                            (comparativo[cp].isna() | (comparativo[cp] == ''))
          distintos       = (comparativo[cp].notna() & (comparativo[cp] != '')) & \
                            (comparativo[cd].notna() & (comparativo[cd] != '')) & \
                            (v_prod_norm != v_dev_norm)

          # Una llave se marca como distinta solo si NINGUNA combinación coincide
          ids_distintos_sin_match = set(comparativo.loc[distintos, col_llave]) - ids_con_match

          def registrar_opcion4(ids_fallo, sufijo, msg_estado):
              if ids_fallo:
                  for id_val in ids_fallo:
                      registros_errores.append({col_llave: id_val, 'Error': f"{col}_{sufijo}", 'Flag': 1})
                  # Ejemplo: tomamos la primera fila del primer id para el print
                  ej_id  = next(iter(ids_fallo))
                  ej_idx = comparativo[comparativo[col_llave] == ej_id].index[0]
                  val_p  = forzar_int_str(comparativo.loc[ej_idx, cp])
                  val_d  = forzar_int_str(comparativo.loc[ej_idx, cd])
                  print(f"{col:<30} | {msg_estado:<20} | {len(ids_fallo)} llaves (Ej ID {ej_id}: {val_p} vs {val_d})")

          # Falta en DEV / PROD: llaves donde TODAS las combinaciones tienen ese vacío
          ids_falta_dev  = set(comparativo.loc[solo_prod_tiene, col_llave]) - ids_con_match
          ids_falta_prod = set(comparativo.loc[solo_dev_tiene,  col_llave]) - ids_con_match

          registrar_opcion4(ids_falta_dev,              "FALTA_EN_DEV", "Falta en DEV")
          registrar_opcion4(ids_falta_prod,             "FALTA_EN_PROD", "Falta en PROD")
          registrar_opcion4(ids_distintos_sin_match,    "DIFERENTE",     "Valores distintos")

  else:
      # Comportamiento original (sin duplicados)
      comparativo = pd.merge(prod_comun, dev_comun, on=col_llave, suffixes=('_prod', '_dev'))

      for col in columnas_a_comparar:
          cp, cd = f"{col}_prod", f"{col}_dev"

          v_prod_norm = comparativo[cp].apply(forzar_int_str)
          v_dev_norm  = comparativo[cd].apply(forzar_int_str)

          solo_prod_tiene = comparativo[cp].notna() & (comparativo[cp] != '') & \
                            (comparativo[cd].isna() | (comparativo[cd] == ''))
          solo_dev_tiene  = comparativo[cd].notna() & (comparativo[cd] != '') & \
                            (comparativo[cp].isna() | (comparativo[cp] == ''))
          distintos       = (comparativo[cp].notna() & (comparativo[cp] != '')) & \
                            (comparativo[cd].notna() & (comparativo[cd] != '')) & \
                            (v_prod_norm != v_dev_norm)

          def registrar(mascara, sufijo, msg_estado):
              if mascara.any():
                  ids_fallo = comparativo.loc[mascara, col_llave].tolist()
                  for id_val in ids_fallo:
                      registros_errores.append({col_llave: id_val, 'Error': f"{col}_{sufijo}", 'Flag': 1})
                  n   = len(ids_fallo)
                  idx = mascara.idxmax()
                  val_p = forzar_int_str(comparativo.loc[idx, cp])
                  val_d = forzar_int_str(comparativo.loc[idx, cd])
                  print(f"{col:<30} | {msg_estado:<20} | {n} filas (Ej ID {comparativo.loc[idx, col_llave]}: {val_p} vs {val_d})")

          registrar(solo_prod_tiene, "FALTA_EN_DEV",  "Falta en DEV")
          registrar(solo_dev_tiene,  "FALTA_EN_PROD", "Falta en PROD")
          registrar(distintos,       "DIFERENTE",      "Valores distintos")

  # 4. CREACIÓN DEL DATAFRAME FINAL DE FLAGS
  if registros_errores:
      df_temp = pd.DataFrame(registros_errores)
      df_ids_diferencias = df_temp.pivot_table(
          index=col_llave,
          columns='Error',
          values='Flag',
          fill_value=0
      ).reset_index()

      cols_error = [c for c in df_ids_diferencias.columns if c != col_llave]
      df_ids_diferencias[cols_error] = df_ids_diferencias[cols_error].astype(int)
  else:
      df_ids_diferencias = pd.DataFrame(columns=[col_llave])

  print("\n" + "="*30)
  print("PROCESO TERMINADO")
  print(f"Registros en df_ids_diferencias: {len(df_ids_diferencias)}")

In [ ]:
if opcion >= 6:
    # ============================================================
    # OPCIONES > 5: LLAVE COMPUESTA (date + id_am)
    # Iteración fecha por fecha para evitar desbordamiento de memoria
    # ============================================================

    COL_DATE = 'date'
    COL_IDAM = 'id_am'

    def preparar_df(df):
        df = df.copy()
        df[COL_DATE] = df[COL_DATE].astype(str).str.strip()
        df[COL_IDAM] = df[COL_IDAM].apply(forzar_int_str)
        return df

    prod = preparar_df(prod)
    dev  = preparar_df(dev)

    # ----------------------------------------------------------
    # 1. SEPARAR CON id_am PRESENTE VS VACÍO
    # ----------------------------------------------------------
    prod_con = prod[prod[COL_IDAM] != ''].copy()
    prod_vac = prod[prod[COL_IDAM] == ''].copy()
    dev_con  = dev[dev[COL_IDAM]  != ''].copy()
    dev_vac  = dev[dev[COL_IDAM]  == ''].copy()

    def llave_str(df):
        return df[COL_DATE].astype(str) + '||' + df[COL_IDAM].astype(str)

    counter_prod_con = Counter(llave_str(prod_con))
    counter_dev_con  = Counter(llave_str(dev_con))
    counter_prod_vac = Counter(prod_vac[COL_DATE])
    counter_dev_vac  = Counter(dev_vac[COL_DATE])

    llaves_prod_todas = llave_str(prod_con).tolist() + [f"{d}||" for d in prod_vac[COL_DATE]]
    llaves_dev_todas  = llave_str(dev_con).tolist()  + [f"{d}||" for d in dev_vac[COL_DATE]]
    dups_prod = pd.Series(llaves_prod_todas).duplicated().sum()
    dups_dev  = pd.Series(llaves_dev_todas).duplicated().sum()
    print(f"Duplicados en prod: {dups_prod}")
    print(f"Duplicados en dev:  {dups_dev}")

    registros_errores = []

    # ----------------------------------------------------------
    # 2. IDENTIFICAR ÚNICOS (EXISTENCIA)
    # ----------------------------------------------------------
    solo_prod_con = list((counter_prod_con - counter_dev_con).elements())
    solo_dev_con  = list((counter_dev_con  - counter_prod_con).elements())

    for llave in solo_prod_con:
        registros_errores.append({'llave': llave, 'Error': 'SOLO_EN_PROD', 'Flag': 1})
    for llave in solo_dev_con:
        registros_errores.append({'llave': llave, 'Error': 'SOLO_EN_DEV', 'Flag': 1})

    solo_prod_vac = list((counter_prod_vac - counter_dev_vac).elements())
    solo_dev_vac  = list((counter_dev_vac  - counter_prod_vac).elements())

    for dia in solo_prod_vac:
        registros_errores.append({'llave': f"{dia}||VACIO_EXTRA", 'Error': 'SOLO_EN_PROD', 'Flag': 1})
    for dia in solo_dev_vac:
        registros_errores.append({'llave': f"{dia}||VACIO_EXTRA", 'Error': 'SOLO_EN_DEV', 'Flag': 1})

    total_solo_prod = len(solo_prod_con) + len(solo_prod_vac)
    total_solo_dev  = len(solo_dev_con)  + len(solo_dev_vac)
    esta_contenido  = total_solo_prod == 0
    nombre_menor = "prod" if len(llaves_prod_todas) <= len(llaves_dev_todas) else "dev"
    nombre_mayor = "dev"  if nombre_menor == "prod" else "prod"

    print(f"\nResultados de pertenencia:")
    print(f"¿La totalidad de {nombre_menor} está contenida en {nombre_mayor}?: {esta_contenido}")
    print(f"\nRegistros en prod que no están en dev: {total_solo_prod}")
    print(f"Registros en dev que no están en prod: {total_solo_dev}")
    if solo_prod_con:
        print(f"Ejemplos de llaves solo en prod: {sorted(solo_prod_con)[:5]}")
    if solo_dev_con:
        print(f"Ejemplos de llaves solo en dev:  {sorted(solo_dev_con)[:5]}")

    # ----------------------------------------------------------
    # 3. PREPARAR ÍNDICES POR FECHA
    # ----------------------------------------------------------
    prod_con['_llave'] = llave_str(prod_con)
    dev_con['_llave']  = llave_str(dev_con)

    llaves_comunes_con = set(counter_prod_con.keys()) & set(counter_dev_con.keys())
    dias_comunes_vac   = set(counter_prod_vac.keys()) & set(counter_dev_vac.keys())

    # Todas las fechas únicas a iterar
    todas_las_fechas = sorted(
        set(prod_con[COL_DATE]) | set(dev_con[COL_DATE]) |
        set(prod_vac[COL_DATE]) | set(dev_vac[COL_DATE])
    )

    columnas_a_comparar = [
        c for c in prod.columns
        if c not in (COL_DATE, COL_IDAM, '_llave') and c in dev.columns
    ]

    print(f"\nFechas a procesar: {len(todas_las_fechas)}")
    print(f"{'COLUMNA':<30} | {'ESTADO':<20} | {'DETALLE'}")
    print("-" * 110)

    # ----------------------------------------------------------
    # 4. ITERAR FECHA POR FECHA
    # ----------------------------------------------------------
    for dia in todas_las_fechas:

        # ---- 4A. CON id_am PRESENTE ----
        p_con_dia = prod_con[(prod_con[COL_DATE] == dia) & (prod_con['_llave'].isin(llaves_comunes_con))]
        d_con_dia = dev_con[(dev_con[COL_DATE]   == dia) & (dev_con['_llave'].isin(llaves_comunes_con))]

        if not p_con_dia.empty and not d_con_dia.empty:
            comp = pd.merge(p_con_dia, d_con_dia, on='_llave', suffixes=('_prod', '_dev'), how='inner')

            for col in columnas_a_comparar:
                cp, cd = f"{col}_prod", f"{col}_dev"

                v_p = comp[cp].apply(forzar_int_str)
                v_d = comp[cd].apply(forzar_int_str)

                coincide      = v_p == v_d
                ids_con_match = set(comp.loc[coincide, '_llave'])

                solo_prod_tiene = comp[cp].notna() & (comp[cp] != '') & \
                                  (comp[cd].isna() | (comp[cd] == ''))
                solo_dev_tiene  = comp[cd].notna() & (comp[cd] != '') & \
                                  (comp[cp].isna() | (comp[cp] == ''))
                distintos       = (comp[cp].notna() & (comp[cp] != '')) & \
                                  (comp[cd].notna() & (comp[cd] != '')) & \
                                  (v_p != v_d)

                ids_falta_dev           = set(comp.loc[solo_prod_tiene, '_llave']) - ids_con_match
                ids_falta_prod          = set(comp.loc[solo_dev_tiene,  '_llave']) - ids_con_match
                ids_distintos_sin_match = set(comp.loc[distintos,       '_llave']) - ids_con_match

                def registrar_con(ids_fallo, sufijo, msg_estado):
                    if ids_fallo:
                        for id_val in ids_fallo:
                            registros_errores.append({'llave': id_val, 'Error': f"{col}_{sufijo}", 'Flag': 1})
                        ej_id  = next(iter(ids_fallo))
                        ej_idx = comp[comp['_llave'] == ej_id].index[0]
                        val_p  = forzar_int_str(comp.loc[ej_idx, cp])
                        val_d  = forzar_int_str(comp.loc[ej_idx, cd])
                        print(f"{col:<30} | {msg_estado:<20} | {len(ids_fallo)} llaves "
                              f"(Ej {ej_id}: {val_p} vs {val_d})")

                registrar_con(ids_falta_dev,           "FALTA_EN_DEV",  "Falta en DEV")
                registrar_con(ids_falta_prod,          "FALTA_EN_PROD", "Falta en PROD")
                registrar_con(ids_distintos_sin_match, "DIFERENTE",     "Valores distintos")

        # ---- 4B. VACÍOS — sin merge, solo sets ----
        if dia in dias_comunes_vac:
            filas_p = prod_vac[prod_vac[COL_DATE] == dia]
            filas_d = dev_vac[dev_vac[COL_DATE]   == dia]

            for col in columnas_a_comparar:
                vals_p = filas_p[col].apply(forzar_int_str)
                vals_d = filas_d[col].apply(forzar_int_str)

                set_p = set(vals_p)
                set_d = set(vals_d)

                hay_match          = bool((set_p & set_d) - {''})
                prod_tiene_dev_vac = (vals_p != '').any() and (vals_d == '').all()
                dev_tiene_prod_vac = (vals_d != '').any() and (vals_p == '').all()
                solo_en_p          = set_p - set_d - {''}
                llave_reporte      = f"{dia}||VACIO"

                if prod_tiene_dev_vac and not hay_match:
                    registros_errores.append({'llave': llave_reporte,
                                              'Error': f"{col}_FALTA_EN_DEV", 'Flag': 1})
                    ej = next(iter(set_p - {''}), '')
                    print(f"{col:<30} | {'Falta en DEV':<20} | día {dia} (prod: {ej} vs dev: vacío)")

                elif dev_tiene_prod_vac and not hay_match:
                    registros_errores.append({'llave': llave_reporte,
                                              'Error': f"{col}_FALTA_EN_PROD", 'Flag': 1})
                    ej = next(iter(set_d - {''}), '')
                    print(f"{col:<30} | {'Falta en PROD':<20} | día {dia} (prod: vacío vs dev: {ej})")

                elif solo_en_p and not hay_match:
                    ej_p = next(iter(solo_en_p))
                    ej_d = next(iter(set_d - {''}), '')
                    registros_errores.append({'llave': llave_reporte,
                                              'Error': f"{col}_DIFERENTE", 'Flag': 1})
                    print(f"{col:<30} | {'Valores distintos':<20} | día {dia} "
                          f"(Ej prod: {ej_p} vs dev: {ej_d})")

    # ----------------------------------------------------------
    # 5. DATAFRAME FINAL DE FLAGS
    # ----------------------------------------------------------
    if registros_errores:
        df_temp = pd.DataFrame(registros_errores)
        df_ids_diferencias = df_temp.pivot_table(
            index='llave',
            columns='Error',
            values='Flag',
            fill_value=0
        ).reset_index()

        cols_error = [c for c in df_ids_diferencias.columns if c != 'llave']
        df_ids_diferencias[cols_error] = df_ids_diferencias[cols_error].astype(int)
    else:
        df_ids_diferencias = pd.DataFrame(columns=['llave'])

    print("\n" + "=" * 30)
    print("PROCESO TERMINADO")
    print(f"Registros en df_ids_diferencias: {len(df_ids_diferencias)}")

In [ ]:
if opcion == 2:
  # Igualar tipo de sf_order_id en todos los dfs
  prod['sf_order_id'] = prod['sf_order_id'].astype(str)
  dev['sf_order_id']  = dev['sf_order_id'].astype(str)
  df_ids_diferencias['sf_order_id'] = df_ids_diferencias['sf_order_id'].astype(str)

  prod['fecha_de_creacion'] = pd.to_datetime(prod['fecha_de_creacion'])
  dev['fecha_de_creacion']  = pd.to_datetime(dev['fecha_de_creacion'])

  merged = prod[['sf_order_id', 'fecha_de_creacion']].merge(
      dev[['sf_order_id', 'fecha_de_creacion']],
      on='sf_order_id',
      suffixes=('_prod', '_dev')
  )

  merged['fecha_creacion_dif_prod_menos_dev'] = (
      merged['fecha_de_creacion_prod'] -
      merged['fecha_de_creacion_dev']
  ).dt.days

  df_ids_diferencias = df_ids_diferencias.merge(
      merged[['sf_order_id', 'fecha_creacion_dif_prod_menos_dev']],
      on='sf_order_id',
      how='left'
  )

In [ ]:
# update_sheets_in_drive_folder(gc, '15SPKEg41SyiDNWb06WkTi0cJj8Sn9Vcx9F1uszdV2Ik', 'Data', df_ids_diferencias)

print(df_ids_diferencias.shape)

#
update_sheets_in_drive_folder_chunked(gc, '15SPKEg41SyiDNWb06WkTi0cJj8Sn9Vcx9F1uszdV2Ik', hoja_escritura, df_ids_diferencias)